# Carrega e separa imagens do dataset


## Helpers


In [1]:
import os
import zipfile
import io
from PIL import Image

def converter_docx_para_png(pasta_raiz, deletar_docx_original=False):
  """
  Percorre pasta_raiz (em qualquer profundidade) e converte cada .docx
  encontrado num .png de verdade, salvo NA MESMA PASTA onde o .docx está
  (não reorganiza a estrutura de dias/subpastas que já existe).

  deletar_docx_original: se True, apaga o .docx depois de converter com
  sucesso. Por padrão fica False (mais seguro) — os dois arquivos convivem
  na mesma pasta até você conferir que a conversão saiu certa.
  """
  convertidos = 0
  sem_imagem = []
  falhou_abrir = []

  for raiz, _, arquivos in os.walk(pasta_raiz):
    for nome in arquivos:
      if not nome.lower().endswith('.docx'):
        continue

      caminho_docx = os.path.join(raiz, nome)
      nome_logico, correcoes = sanitizar_nome(nome)
      nome_base = os.path.splitext(nome_logico)[0]

      try:
        with zipfile.ZipFile(caminho_docx, 'r') as docx_zip:
          imagens_internas = [
              f for f in docx_zip.namelist()
              if f.startswith('word/media/') and f.lower().endswith(('.png', '.jpg', '.jpeg'))
          ]
          if not imagens_internas:
            sem_imagem.append(caminho_docx)
            continue

          maior = max(imagens_internas, key=lambda f: docx_zip.getinfo(f).file_size)
          dados_imagem = docx_zip.read(maior)

          try:
            imagem = Image.open(io.BytesIO(dados_imagem)).convert('RGB')
          except Exception as e:
            falhou_abrir.append((caminho_docx, maior, str(e)))
            continue

          caminho_destino = os.path.join(raiz, f'{nome_base}.png')
          imagem.save(caminho_destino, 'PNG')
          convertidos += 1

          if correcoes:
            print(f'  sanitizado [{", ".join(correcoes)}]: {nome!r} -> {nome_base}.png')

          if deletar_docx_original:
            os.remove(caminho_docx)

      except zipfile.BadZipFile:
        print(f'Aviso: {caminho_docx} não abriu como zip (corrompido?)')

  print(f'\n{convertidos} imagem(ns) convertida(s) para PNG, na mesma pasta do .docx original.')

  if sem_imagem:
    print(f'\n{len(sem_imagem)} docx sem imagem embutida encontrada:')
    for c in sem_imagem:
      print(' ', c)

  if falhou_abrir:
    print(f'\n{len(falhou_abrir)} imagem(ns) que o Pillow não conseguiu abrir:')
    for caminho_docx, imagem_interna, erro in falhou_abrir:
      print(f'  {caminho_docx} -> {imagem_interna} ({erro})')

In [2]:
# Recuperar imagens faltantes

def extrair_imagens_faltantes(pasta_segmentada, pasta_no_segmentada, limiar_preto=30):
  """Extrai dos .docx segmentados as fotos que não têm equivalente na pasta
  não-segmentada (recalculado a cada execução), remove o fundo preto e salva
  como .png junto às demais imagens do dia.

  A presença é checada pela chave completa (C, Dia, Grupo, Concentração
  normalizada): fotos de concentrações diferentes não são mais confundidas
  com duplicatas (caso de \"C2 - D5 0 OC - 1_\")."""
  from PIL import Image
  import numpy as np
  import io

  ext_imagem = ('.jpg', '.jpeg', '.png')
  chaves_presentes = set()
  for raiz, _, arquivos in os.walk(pasta_no_segmentada):
      for nome in arquivos:
          if not nome.lower().endswith(ext_imagem):
              continue
          chave = chave_de(nome)
          if chave:
              chaves_presentes.add(chave)

  recuperadas = 0
  for raiz, _, arquivos in os.walk(pasta_segmentada):
      if not os.path.basename(raiz).startswith('Dia'):
          continue
      for nome in arquivos:
          if not nome.lower().endswith('.docx'):
              continue

          nome_logico, correcoes_aplicadas = sanitizar_nome(nome)
          chave = chave_de(nome_logico)

          if chave is None:
              print(f'  sem match após sanitização (conferir): {nome}')
              continue

          if chave in chaves_presentes:
              continue

          m_dia = re.search(r'D(\d+)', nome_logico, re.IGNORECASE)
          dia_destino = f'Dia {m_dia.group(1)}' if m_dia else os.path.basename(raiz)
          if correcoes_aplicadas:
              print(f'  sanitizado [{", ".join(correcoes_aplicadas)}]: {nome!r} → {nome_logico!r}')
          if dia_destino != os.path.basename(raiz):
              print(f'  alocação corrigida: {nome!r} estava em {os.path.basename(raiz)!r}'
                    f', destino correto: {dia_destino!r}')

          with zipfile.ZipFile(os.path.join(raiz, nome)) as zip_ref:
              medias = [n for n in zip_ref.namelist()
                        if n.startswith('word/media/') and n.endswith('.png')]
              if not medias:
                  print(f'  sem PNG dentro: {nome}')
                  continue
              maior = max(medias, key=lambda n: zip_ref.getinfo(n).file_size)
              img = Image.open(io.BytesIO(zip_ref.read(maior))).convert('RGB')

          arr = np.asarray(img, dtype=np.float32)
          cinza = arr.mean(axis=2)
          ys, xs = np.nonzero(cinza > limiar_preto)
          if ys.size == 0:
              print(f'  foto totalmente preta (sem corte): {nome}')
              continue
          corte = arr[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

          destino_dir = os.path.join(pasta_no_segmentada, dia_destino)
          os.makedirs(destino_dir, exist_ok=True)
          nome_saida = os.path.splitext(nome_logico)[0] + '.png'
          destino = os.path.join(destino_dir, nome_saida)
          Image.fromarray(corte.astype(np.uint8)).save(destino)
          chaves_presentes.add(chave)
          recuperadas += 1
          print(f'  recuperada: {os.path.join(dia_destino, nome_saida)}')

  print(f'{recuperadas} foto(s) recuperada(s) dos .docx segmentados para \'{pasta_no_segmentada}\'.')

In [3]:
# Selftest

def _selftest():
  """Valida o contrato de parsing/normalização (sem framework)."""
  casos = {
      'C2 - D5 - SEMO- 0_.docx': ('2', '5', 'SEMO', '0'),
      'C1 - D0- OT- 08_.jpg':    ('1', '0', 'OT', '08'),
      'C1-D0-OT-0,4_.jpg':       ('1', '0', 'OT', '04'),
      'C2- D5- OO - 04_.png':    ('2', '5', 'OO', '04'),
      'C2 - D5 - 00 - 1_.png':   ('2', '5', 'OO', '1'),
      'C2 - D5 0 OC - 1_.docx':  ('2', '5', 'OC', '1'),
      'C2 - D6 - 0T - 08_.png':   ('2', '6', 'OT', '08'),
      'C2- D5 - OC - 08_.png':   ('2', '5', 'OC', '08'),
      'C1 - D4 - SEMO - 0_.jpg': ('1', '4', 'SEMO', '0'),
  }
  for nome, esperado in casos.items():
      obtido = chave_de(nome)
      assert obtido == esperado, f'{nome!r}: {obtido} != {esperado}'

  tokens = {
      '04': '04', '0,4': '04', '4': '04', '0.4': '04',
      '08': '08', '8': '08', '1': '1', '01': '1', '0': '0',
  }
  for bruto, esperado in tokens.items():
      assert normalizar_token(bruto) == esperado, f'token {bruto!r}'

  presentes = {('2', '5', 'OC', '04'), ('2', '5', 'OC', '08')}
  assert ('2', '5', 'OC', '1') not in presentes, (
      'foto de concentração diferente NÃO deve ser confundida com duplicata')
  assert ('2', '5', 'OC', '04') in presentes
  print('selftest: ok')

In [4]:
# Montar chave composta

import re

def normalizar_token(token):
  """Colapsa variações de concentração: 04/0,4/4/0.4 → 04, 08/8 → 08, 1/01 → 1."""
  t = token.replace(',', '.')
  try:
      v = float(t)
  except ValueError:
      return t
  if v == 1.0:
      return '1'
  if v in (0.4, 4.0):
      return '04'
  if v in (0.8, 8.0):
      return '08'
  return t


PADRAO_CHAVE_EXT = re.compile(
    r'C(\d+)\s*-\s*D(\d+)\s*-\s*([A-Za-z0-9]+)\s*-\s*([A-Za-z0-9,]+)_',
    re.IGNORECASE,
)

def chave_de(nome):
  """Extrai (C, Dia, Grupo, Concentração normalizada) do nome já sanitizado, ou None."""
  nome_san, _ = sanitizar_nome(nome)
  m = PADRAO_CHAVE_EXT.search(nome_san)
  if not m:
      return None
  return (m.group(1), m.group(2), m.group(3).upper(), normalizar_token(m.group(4)))

In [5]:
# Auditoria e Separacao

def listar_codigos_encontrados(pasta_origem):
  """Confere todos os códigos de grupo encontrados em cada pasta Dia X."""
  codigos = set()
  sem_match = []

  for raiz, _, arquivos in os.walk(pasta_origem):
      for nome in arquivos:
          if not nome.lower().endswith(EXTENSOES_VALIDAS):
              continue
          codigo = extrair_codigo(nome)
          if codigo:
              codigos.add(codigo)
          else:
              sem_match.append(os.path.join(raiz, nome))

  print('Códigos de grupo encontrados no dataset todo:', sorted(codigos))
  if sem_match:
      print('\nArquivos que não bateram com o regex (conferir manualmente):')
      for caminho in sem_match:
          print(' ', caminho)

def separar_controle(pasta_origem, pasta_destino, codigo_controle):
  copiados = 0
  ignorados = 0

  for raiz, _, arquivos in os.walk(pasta_origem):
      for nome in arquivos:
          if not nome.lower().endswith(EXTENSOES_VALIDAS):
              continue

          codigo = extrair_codigo(nome)
          if codigo == codigo_controle.upper():
              subpasta_relativa = os.path.relpath(raiz, pasta_origem)
              destino_dir = os.path.join(pasta_destino, subpasta_relativa)
              os.makedirs(destino_dir, exist_ok=True)

              shutil.copy2(os.path.join(raiz, nome), os.path.join(destino_dir, nome))
              copiados += 1
          else:
              ignorados += 1

  print(f"{copiados} foto(s) de controle ({codigo_controle}) copiada(s) para '{pasta_destino}'.")
  print(f'{ignorados} foto(s) de grupos com óleo (ou não reconhecidas) foram ignoradas.')

In [6]:
# Sanitizar e Extrair Codigo

def sanitizar_nome(nome_arquivo):
  """Retorna o nome canônico corrigido e a lista de correções aplicadas.
  O nome em disco não é alterado, a correção é aplicada apenas ao nome
  lógico usado para lookup de chaves e para nomear o arquivo de saída."""
  nome = nome_arquivo
  correcoes = []

  # 1. Separador ausente antes do grupo: "C2 - D5 0 OC - 1_" → "C2 - D5 - OC - 1_"
  nome, n = re.subn(
      r'(C\d+\s*-\s*D\d+)\s+\d*\s*([A-Za-z]{2,4})\s+-',
      r'\1 - \2 -',
      nome,
  )
  if n:
      correcoes.append('separador ausente antes do grupo')

  # 2a. Código de grupo com zero inicial: "- 0T -" → "- OT -"
  nome, n = re.subn(r'(?<=-\s)0([TOACtoac])(?=\s*-)', r'O\1', nome)
  if n:
      correcoes.append('0X→OX no código de grupo')

  # 2b. Código de grupo duplo zero: "- 00 -" → "- OO -"
  nome, n = re.subn(r'(?<=-\s)00(?=\s*-)', 'OO', nome)
  if n:
      correcoes.append('00→OO no código de grupo')

  # 3. Letra O no campo de concentração: "- SEMO - O_" → "- SEMO - 0_"
  nome, n = re.subn(r'(-\s*[A-Z]+\s*-\s*)O(_)', r'\g<1>0\2', nome, flags=re.IGNORECASE)
  if n:
      correcoes.append('O→0 na concentração')

  # 4. Vírgula decimal na concentração: "0,4_" → "04_"
  nome, n = re.subn(r'\b0,(\d)_', r'0\1_', nome)
  if n:
      correcoes.append('vírgula decimal na concentração')

  # 5. Concentração com dígito único: "- 4_" / "- 8_" → "- 04_" / "- 08_"
  nome, n = re.subn(r'(?<=-\s)([48])_', r'0\1_', nome)
  if n:
      correcoes.append('zero à esquerda na concentração')

  return nome, correcoes

def extrair_codigo(nome_arquivo):
  """Verifica se o nome do arquivo contém alguma variação do código de grupo."""
  nome_sanitizado, _ = sanitizar_nome(nome_arquivo)
  match = PADRAO_REGEX.search(nome_sanitizado)

  return match.group(1).upper() if match else None

In [7]:
import os

CAMINHO_DATA_RAW = '/content/data-raw'

if not os.path.exists(CAMINHO_DATA_RAW):
  os.makedirs(CAMINHO_DATA_RAW)
  print(f'Pasta criada: {CAMINHO_DATA_RAW}')
else:
  print(f'Pasta já existe, nada foi feito: {CAMINHO_DATA_RAW}')


Pasta criada: /content/data-raw


## Carregar Datasets

In [8]:
# Baixar .zip não segmentado

from google.colab import drive
import zipfile
import shutil
import os

CAMINHO_NO_SEGMENTED_DATA = '/content/data-raw/no-segmented'
if not os.path.exists(CAMINHO_NO_SEGMENTED_DATA + '/all'):
  drive.mount('/content/drive')
  os.makedirs(CAMINHO_NO_SEGMENTED_DATA)
  print(f'Pasta criada: {CAMINHO_NO_SEGMENTED_DATA}')
else:
  print(f'Pasta já existe, nada foi feito: {CAMINHO_NO_SEGMENTED_DATA}')

if not os.path.exists(f'{CAMINHO_NO_SEGMENTED_DATA}/dataset_tilapias'):
  drive.mount('/content/drive')
  zip_imagens = '/content/drive/MyDrive/PIC Tilápia/Dataset/dataset_tilapias_compactado.zip'
  shutil.copy(zip_imagens, f'{CAMINHO_NO_SEGMENTED_DATA}/dataset_tilapias.zip')

  with zipfile.ZipFile(f'{CAMINHO_NO_SEGMENTED_DATA}/dataset_tilapias.zip', 'r') as zip_ref:
    zip_ref.extractall(f'{CAMINHO_NO_SEGMENTED_DATA}/all')

Mounted at /content/drive
Pasta criada: /content/data-raw/no-segmented
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Baixar segmentado

from google.colab import drive
import zipfile
import shutil
import os

CAMINHO_SEGMENTED_DATA = '/content/data-raw/segmented'
if not os.path.exists(CAMINHO_SEGMENTED_DATA + '/all'):
  drive.mount('/content/drive')
  os.makedirs(CAMINHO_SEGMENTED_DATA)
  print(f'Pasta criada: {CAMINHO_SEGMENTED_DATA}')
else:
  print(f'Pasta já existe, nada foi feito: {CAMINHO_SEGMENTED_DATA}')

if not os.path.exists(f'{CAMINHO_SEGMENTED_DATA}/dataset_tilapias_segmentado'):
  drive.mount('/content/drive')
  zip_imagens = '/content/drive/MyDrive/PIC Tilápia/Dataset/dataset_tilapias_segmentado_compactado.zip'
  shutil.copy(zip_imagens, f'{CAMINHO_SEGMENTED_DATA}/dataset_tilapias_segmentado.zip')

  with zipfile.ZipFile(f'{CAMINHO_SEGMENTED_DATA}/dataset_tilapias_segmentado.zip', 'r') as zip_ref:
    zip_ref.extractall(f'{CAMINHO_SEGMENTED_DATA}/all')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta criada: /content/data-raw/segmented
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Processar Segmentado

In [10]:
# Config Segmentado

import os
import re
import shutil

PASTA_ORIGEM_SEGMENTADOS = f'{CAMINHO_SEGMENTED_DATA}/all'
PASTA_CONTROLE_SEGMENTADOS = f'{CAMINHO_SEGMENTED_DATA}/control'

CODIGO_CONTROLE = 'SEMO'

PADRAO_REGEX = re.compile(r'C\d+\s*-\s*D\d+\s*-\s*([A-Za-z]+)\s*-', re.IGNORECASE)

EXTENSOES_VALIDAS = ('.docx')

In [11]:
# Auditoria Codigos Segmentados

listar_codigos_encontrados(PASTA_ORIGEM_SEGMENTADOS)

Códigos de grupo encontrados no dataset todo: ['OA', 'OC', 'OO', 'OT', 'SEMO']


In [12]:
# Separar Controle Segmentados

separar_controle(PASTA_ORIGEM_SEGMENTADOS, PASTA_CONTROLE_SEGMENTADOS, CODIGO_CONTROLE)

22 foto(s) de controle (SEMO) copiada(s) para '/content/data-raw/segmented/control'.
264 foto(s) de grupos com óleo (ou não reconhecidas) foram ignoradas.


In [13]:
converter_docx_para_png(f'{CAMINHO_SEGMENTED_DATA}/control', deletar_docx_original=False)

  sanitizado [O→0 na concentração]: 'C2 - D4 - SEMO - O_.docx' -> C2 - D4 - SEMO - 0_.png

22 imagem(ns) convertida(s) para PNG, na mesma pasta do .docx original.


In [17]:
converter_docx_para_png(f'{CAMINHO_SEGMENTED_DATA}/all', deletar_docx_original=False)

  sanitizado [separador ausente antes do grupo]: 'C2 - D5 0 OC - 1_.docx' -> C2 - D5 - OC - 1_.png
  sanitizado [00→OO no código de grupo]: 'C2 - D5 - 00 - 1_.docx' -> C2 - D5 - OO - 1_.png
  sanitizado [O→0 na concentração]: 'C2 - D4 - SEMO - O_.docx' -> C2 - D4 - SEMO - 0_.png
  sanitizado [0X→OX no código de grupo]: 'C2 - D6 - 0T - 08_.docx' -> C2 - D6 - OT - 08_.png
  sanitizado [zero à esquerda na concentração]: 'C2 - D6 - OC - 8_.docx' -> C2 - D6 - OC - 08_.png
  sanitizado [zero à esquerda na concentração]: 'C2 - D6 - OT - 4_.docx' -> C2 - D6 - OT - 04_.png
  sanitizado [zero à esquerda na concentração]: 'C2 - D6 - OA - 8_.docx' -> C2 - D6 - OA - 08_.png
  sanitizado [zero à esquerda na concentração]: 'C1 - D2 - OA - 8_.docx' -> C1 - D2 - OA - 08_.png
  sanitizado [zero à esquerda na concentração]: 'C2 - D2 - OA - 8_.docx' -> C2 - D2 - OA - 08_.png
  sanitizado [vírgula decimal na concentração]: 'C1-D0-OT-0,4_.docx' -> C1-D0-OT-04_.png
  sanitizado [zero à esquerda na concentraç

## Processar Não Segmentado

In [14]:
# Config Não Segmentado

import os
import re
import shutil

PASTA_ORIGEM = f'{CAMINHO_NO_SEGMENTED_DATA}/all'
PASTA_CONTROLE = f'{CAMINHO_NO_SEGMENTED_DATA}/control'

CODIGO_CONTROLE = 'SEMO'

PADRAO_REGEX = re.compile(r'C\d+\s*-\s*D\d+\s*-\s*([A-Za-z]+)\s*-', re.IGNORECASE)
PADRAO_CHAVE = re.compile(r'C(\d+)\s*-\s*D(\d+)\s*-\s*([A-Za-z0-9]+)\s*-', re.IGNORECASE)

EXTENSOES_VALIDAS = ('.jpg', '.jpeg', '.png')

In [15]:
# Auditoria Codigos Não Segmentados

listar_codigos_encontrados(PASTA_ORIGEM)

Códigos de grupo encontrados no dataset todo: ['OA', 'OC', 'OO', 'OT', 'SEMO']


In [16]:
# Recuperar e Separar Controle

extrair_imagens_faltantes(PASTA_ORIGEM_SEGMENTADOS, PASTA_ORIGEM)
separar_controle(PASTA_ORIGEM, PASTA_CONTROLE, CODIGO_CONTROLE)

  recuperada: Dia 5/C2 - D5 - OO - 08_.png
  recuperada: Dia 5/C2 - D5 - OA - 04_.png
  recuperada: Dia 5/C2 - D5 - OT - 1_.png
  recuperada: Dia 5/C2 - D5- OO - 04_.png
  recuperada: Dia 5/C2 - D5 - SEMO- 0_.png
  recuperada: Dia 5/C2 - D5 - OA - 1_.png
  recuperada: Dia 5/C2 - D5 - OC - 04_.png
  recuperada: Dia 5/C2 - D5 - OT - 08_.png
  sanitizado [separador ausente antes do grupo]: 'C2 - D5 0 OC - 1_.docx' → 'C2 - D5 - OC - 1_.docx'
  recuperada: Dia 5/C2 - D5 - OC - 1_.png
  recuperada: Dia 5/C2- D5- OT - 04_.png
  recuperada: Dia 5/C2- D5 - OC - 08_.png
  recuperada: Dia 5/C2 - D5 - OA - 08_.png
  sanitizado [00→OO no código de grupo]: 'C2 - D5 - 00 - 1_.docx' → 'C2 - D5 - OO - 1_.docx'
  recuperada: Dia 5/C2 - D5 - OO - 1_.png
  recuperada: Dia 4/C1 - D4 - OC - 04_.png
  recuperada: Dia 4/C1 - D4 - OO - 1_.png
  recuperada: Dia 4/C1 - D4 - OT - 04_.png
  recuperada: Dia 4/C1 - D4 - OA - 1_.png
  recuperada: Dia 4/C1 - D4 - SEMO - 0_.png
  recuperada: Dia 4/C1 - D4 - OT - 1_.png

# Features - RGB

In [24]:
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd


# Pasta contendo os dias e suas imagens
pasta = Path("/content/data-raw/segmented/all")

resultados = []

for caminho in sorted(pasta.rglob("*")):
    # Processa somente imagens
    if caminho.suffix.lower() not in (".png", ".jpg", ".jpeg"):
        continue

    if "SEMO" not in caminho.name.upper():
        continue

    imagem = Image.open(caminho).convert("RGB")
    pixels = np.array(imagem)

    # Ignora o fundo totalmente preto
    mascara = np.any(pixels > 0, axis=2)
    pixels_file = pixels[mascara]

    media_r, media_g, media_b = pixels_file.mean(axis=0)

    # Adiciona uma linha à lista
    resultados.append({
        "arquivo": str(caminho.relative_to(pasta)),
        "media_r": media_r,
        "media_g": media_g,
        "media_b": media_b,
    })

# Transforma a lista em uma tabela
df = pd.DataFrame(
    resultados,
    columns=["arquivo", "media_r", "media_g", "media_b"]
)

# Salva a tabela
saida = "/content/features_cores.csv"
df.to_csv(saida, index=False)

print(f"Total de imagens SEMO: {len(df)}")
print(df.to_string(index=False))

print(f"{len(df)} imagens processadas.")
display(df)

Total de imagens SEMO: 22
                              arquivo    media_r    media_g    media_b
        Fotos/Dia 0/C1-D0-SEMO-0_.png 143.763981 144.441422 144.764883
       Fotos/Dia 0/C2-D0- SEMO-0_.png 142.860175 143.070470 143.994300
   Fotos/Dia 1/C1 - D1 - SEMO- 0_.png 147.785709 149.755166 150.402872
   Fotos/Dia 1/C2 - D1- SEMO - 0_.png 139.085142 141.077884 143.245656
Fotos/Dia 10/C1 - D10 - SEMO - 0_.png 141.589024 138.429918 138.693185
 Fotos/Dia 10/C2 - D10 - SEMO- 0_.png 141.464947 136.323937 135.650586
   Fotos/Dia 2/C1 - D2- SEMO - 0_.png 144.171231 146.023780 145.912991
  Fotos/Dia 2/C2 - D2 - SEMO - 0_.png 141.325403 142.426444 143.631368
   Fotos/Dia 3/C1 - D3 - SEMO- 0_.png 144.066242 144.956713 144.105384
  Fotos/Dia 3/C2 - D3 - SEMO - 0_.png 141.230993 141.276999 140.408319
  Fotos/Dia 4/C1 - D4 - SEMO - 0_.png 143.930132 145.147383 144.386177
  Fotos/Dia 4/C2 - D4 - SEMO - 0_.png 138.793366 139.647071 139.715441
   Fotos/Dia 5/C1 - D5 - SEMO- 0_.png 143.019899 14

,arquivo,media_r,media_g,media_b
0,Fotos/Dia 0/C1-D0-SEMO-0_.png,143.763981,144.441422,144.764883
1,Fotos/Dia 0/C2-D0- SEMO-0_.png,142.860175,143.070470,143.994300
2,Fotos/Dia 1/C1 - D1 - SEMO- 0_.png,147.785709,149.755166,150.402872
3,Fotos/Dia 1/C2 - D1- SEMO - 0_.png,139.085142,141.077884,143.245656
4,Fotos/Dia 10/C1 - D10 - SEMO - 0_.png,141.589024,138.429918,138.693185
5,Fotos/Dia 10/C2 - D10 - SEMO- 0_.png,141.464947,136.323937,135.650586
6,Fotos/Dia 2/C1 - D2- SEMO - 0_.png,144.171231,146.023780,145.912991
7,Fotos/Dia 2/C2 - D2 - SEMO - 0_.png,141.325403,142.426444,143.631368
8,Fotos/Dia 3/C1 - D3 - SEMO- 0_.png,144.066242,144.956713,144.105384
9,Fotos/Dia 3/C2 - D3 - SEMO - 0_.png,141.230993,141.276999,140.408319
